# Experiment 05 — Perch Priors and Genus Proxies

**Builds on:** Exp04's tuned Perch LR probe (`C=0.05`, `class_weight=balanced`, probe blend `0.25`).

**New in this experiment:**
1. Fill unmapped competition species with genus-level Perch proxy logits when Perch has other species from the same genus.
2. Add fold-safe site/hour priors parsed from soundscape filenames.

**Hypothesis:** The labeled soundscape set contains strong recorder-site and time-of-day signal, and genus proxies may recover signal for some of the 31 unmapped species without leaving the Perch feature family.

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit as sigmoid, logit
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMB_CACHE = CACHE_DIR / "exp02_embeddings.npy"
LOG_CACHE = CACHE_DIR / "exp02_logits.npy"
RESULTS_CSV = CACHE_DIR / "exp05_priors_proxies_results.csv"
RESULTS_JSON = CACHE_DIR / "exp05_priors_proxies_best.json"

N_SPLITS = 5
PCA_DIM = 64
MIN_POS = 3
BEST_C = 0.05
BEST_CLASS_WEIGHT = "balanced"
BEST_PROBE_BLEND = 0.25
PRIOR_BLEND_GRID = [0.00, 0.025, 0.05, 0.10, 0.20, 0.30]
PROXY_REDUCE_GRID = ["none", "max", "mean"]

print("Project root:", PROJECT_ROOT)
print("Embedding cache exists:", EMB_CACHE.exists())
print("Logit cache exists:", LOG_CACHE.exists())

Project root: /Users/jroessler/Development/kaggle/birdclef
Embedding cache exists: True
Logit cache exists: True


In [2]:
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
N_CLASSES = len(taxonomy)
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
taxon_ids = taxonomy["primary_label"].tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}
class_name = taxonomy.set_index("primary_label")["class_name"].to_dict()

perch_labels = pd.read_csv(DATA_DIR / "models/perch_tf/assets/labels.csv", header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))

mapping = taxonomy.merge(perch_labels, on="scientific_name", how="left")
mapped_mask = mapping["bc_index"].notna().values
comp_to_bc = {
    taxon_to_idx[str(row["primary_label"])] : int(row["bc_index"])
    for _, row in mapping[mapping["bc_index"].notna()].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

def parse_soundscape_filename(filename):
    m = re.search(r"BC2026_Train_\d+_(S\d+)_([0-9]{8})_([0-9]{6})\.ogg", filename)
    if not m:
        return pd.Series({"site": "UNK", "hour_utc": -1, "month": -1})
    site, ymd, hms = m.groups()
    return pd.Series({"site": site, "hour_utc": int(hms[:2]), "month": int(ymd[4:6])})

raw_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"])
    .reset_index(drop=True)
)
labels_df = pd.concat([labels_df, labels_df["filename"].apply(parse_soundscape_filename)], axis=1)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)
emb_full = np.load(EMB_CACHE).astype(np.float32)
logits_full = np.load(LOG_CACHE).astype(np.float32)

groups = labels_df["filename"].values
folds = list(GroupKFold(n_splits=N_SPLITS).split(emb_full, groups=groups))
active_cols = np.where(Y.sum(axis=0) > 0)[0]
unmapped_cols = np.where(~mapped_mask)[0]
active_unmapped_cols = np.intersect1d(active_cols, unmapped_cols)

print(f"Mapped species: {mapped_mask.sum()}/{N_CLASSES}")
print(f"Labels: {Y.shape}, files: {labels_df['filename'].nunique()}, sites: {labels_df['site'].nunique()}, hours: {sorted(labels_df['hour_utc'].unique().tolist())}")
print(f"Active classes: {len(active_cols)}, active unmapped classes: {len(active_unmapped_cols)}")

Mapped species: 203/234
Labels: (739, 234), files: 66, sites: 9, hours: [0, 1, 2, 3, 4, 6, 7, 18, 19, 20, 21, 22, 23]
Active classes: 75, active unmapped classes: 29


In [3]:
def build_scores(proxy_reduce="none"):
    scores = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
    for comp_idx, bc_idx in comp_to_bc.items():
        scores[:, comp_idx] = logits_full[:, bc_idx]

    if proxy_reduce == "none":
        return scores, {}

    proxy_map = {}
    unmapped = mapping[mapping["bc_index"].isna()].copy()
    for _, row in unmapped.iterrows():
        target = str(row["primary_label"])
        scientific_name = str(row["scientific_name"])
        genus = scientific_name.split()[0]
        hits = perch_labels[
            perch_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
        ]
        if hits.empty:
            continue

        comp_idx = taxon_to_idx[target]
        hit_idx = hits["bc_index"].astype(int).values
        if proxy_reduce == "max":
            scores[:, comp_idx] = logits_full[:, hit_idx].max(axis=1)
        elif proxy_reduce == "mean":
            scores[:, comp_idx] = logits_full[:, hit_idx].mean(axis=1)
        else:
            raise ValueError(proxy_reduce)

        proxy_map[target] = {
            "target_scientific_name": scientific_name,
            "class_name": class_name.get(target),
            "genus": genus,
            "n_proxy_species": int(len(hit_idx)),
            "proxy_scientific_names": hits["scientific_name"].head(5).tolist(),
        }

    return scores, proxy_map

for reduce in PROXY_REDUCE_GRID:
    _, proxy_map = build_scores(reduce)
    active_proxy_targets = sorted(set(proxy_map).intersection(set(taxon_ids[i] for i in active_unmapped_cols)))
    by_class = pd.Series([class_name[t] for t in active_proxy_targets]).value_counts().to_dict() if active_proxy_targets else {}
    print(f"proxy_reduce={reduce:<4} proxy_targets={len(proxy_map):2d}, active_proxy_targets={len(active_proxy_targets):2d}, active_by_class={by_class}")

proxy_reduce=none proxy_targets= 0, active_proxy_targets= 0, active_by_class={}
proxy_reduce=max  proxy_targets= 6, active_proxy_targets= 4, active_by_class={'Amphibia': 2, 'Reptilia': 1, 'Mammalia': 1}


proxy_reduce=mean proxy_targets= 6, active_proxy_targets= 4, active_by_class={'Amphibia': 2, 'Reptilia': 1, 'Mammalia': 1}


In [4]:
def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        positives = y_true[:, c].sum()
        if 0 < positives < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)), len(aucs)

def evaluate(name, y_pred, extra=None):
    auc, n_auc = macro_auc(Y, y_pred)
    row = {
        "method": name,
        "macro_auc": auc,
        "auc_classes": n_auc,
        "padded_cmap": padded_cmap(Y, y_pred),
    }
    if extra:
        row.update(extra)
    return row

def fit_prior_tables(meta_df, y_train, strength_site=8.0, strength_hour=8.0, strength_site_hour=4.0):
    global_p = y_train.mean(axis=0).astype(np.float32)
    tables = {"global_p": global_p, "site": {}, "hour": {}, "site_hour": {}}

    for site, idx in meta_df.groupby("site").groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site * global_p) / (len(idx) + strength_site)
        tables["site"][str(site)] = p.astype(np.float32)

    for hour, idx in meta_df.groupby("hour_utc").groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_hour * global_p) / (len(idx) + strength_hour)
        tables["hour"][int(hour)] = p.astype(np.float32)

    for (site, hour), idx in meta_df.groupby(["site", "hour_utc"]).groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site_hour * global_p) / (len(idx) + strength_site_hour)
        tables["site_hour"][(str(site), int(hour))] = p.astype(np.float32)

    return tables

def prior_probs_from_tables(meta_df, tables):
    out = np.repeat(tables["global_p"][None, :], len(meta_df), axis=0).astype(np.float32)
    for i, row in enumerate(meta_df.itertuples(index=False)):
        site = str(row.site)
        hour = int(row.hour_utc)
        if hour in tables["hour"]:
            out[i] = 0.50 * out[i] + 0.50 * tables["hour"][hour]
        if site in tables["site"]:
            out[i] = 0.50 * out[i] + 0.50 * tables["site"][site]
        if (site, hour) in tables["site_hour"]:
            out[i] = 0.35 * out[i] + 0.65 * tables["site_hour"][(site, hour)]
    return np.clip(out, 1e-5, 1.0 - 1e-5)

def build_oof_predictions(scores):
    raw_probs = sigmoid(scores)
    oof_probe = raw_probs.copy()
    oof_prior = np.zeros_like(raw_probs)
    fitted = 0

    for fold, (tr, va) in enumerate(folds):
        scaler = StandardScaler()
        pca = PCA(n_components=PCA_DIM, whiten=True, random_state=42)
        emb_tr = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
        emb_va = pca.transform(scaler.transform(emb_full[va]))
        X_tr = np.concatenate([emb_tr, scores[tr]], axis=1)
        X_va = np.concatenate([emb_va, scores[va]], axis=1)

        active = [c for c in range(N_CLASSES) if MIN_POS <= Y[tr, c].sum() < len(tr)]
        for c in active:
            clf = LogisticRegression(
                C=BEST_C,
                class_weight=BEST_CLASS_WEIGHT,
                max_iter=1000,
                solver="lbfgs",
            )
            clf.fit(X_tr, Y[tr, c])
            oof_probe[va, c] = clf.predict_proba(X_va)[:, 1]
            fitted += 1

        # Fold-safe priors: fit only on training files/windows, predict held-out files/windows.
        train_meta = labels_df.iloc[tr][["site", "hour_utc"]].reset_index(drop=True)
        val_meta = labels_df.iloc[va][["site", "hour_utc"]].reset_index(drop=True)
        tables = fit_prior_tables(train_meta, Y[tr])
        oof_prior[va] = prior_probs_from_tables(val_meta, tables)

    tuned_probe = BEST_PROBE_BLEND * oof_probe + (1.0 - BEST_PROBE_BLEND) * raw_probs
    return raw_probs, oof_probe, tuned_probe, oof_prior, fitted

In [5]:
rows = []
artifacts = {}

for proxy_reduce in PROXY_REDUCE_GRID:
    print(f"\n=== proxy_reduce={proxy_reduce} ===")
    scores, proxy_map = build_scores(proxy_reduce)
    raw_probs, oof_probe, tuned_probe, oof_prior, fitted = build_oof_predictions(scores)

    rows.append(evaluate(
        "raw_perch_scores",
        raw_probs,
        {"proxy_reduce": proxy_reduce, "prior_blend": 0.0, "fitted_probes": 0, "proxy_targets": len(proxy_map)},
    ))
    rows.append(evaluate(
        "tuned_probe_no_prior",
        tuned_probe,
        {"proxy_reduce": proxy_reduce, "prior_blend": 0.0, "fitted_probes": fitted, "proxy_targets": len(proxy_map)},
    ))

    for beta in PRIOR_BLEND_GRID:
        if beta == 0.0:
            continue
        pred = (1.0 - beta) * tuned_probe + beta * oof_prior
        rows.append(evaluate(
            "tuned_probe_prior_blend",
            pred,
            {"proxy_reduce": proxy_reduce, "prior_blend": beta, "fitted_probes": fitted, "proxy_targets": len(proxy_map)},
        ))

    artifacts[proxy_reduce] = {
        "proxy_map": proxy_map,
        "fitted_probes": fitted,
    }
    best_so_far = pd.DataFrame(rows).sort_values(["padded_cmap", "macro_auc"], ascending=False).iloc[0]
    print(f"Best so far: {best_so_far['method']} proxy={best_so_far['proxy_reduce']} prior={best_so_far['prior_blend']} cmAP={best_so_far['padded_cmap']:.4f} AUC={best_so_far['macro_auc']:.4f}")

results = pd.DataFrame(rows).sort_values(["padded_cmap", "macro_auc"], ascending=False).reset_index(drop=True)
results.to_csv(RESULTS_CSV, index=False)

best = results.iloc[0].to_dict()
best["artifacts"] = {
    "proxy_reduce_grid": PROXY_REDUCE_GRID,
    "prior_blend_grid": PRIOR_BLEND_GRID,
    "best_proxy_targets_preview": list(artifacts.get(best["proxy_reduce"], {}).get("proxy_map", {}).items())[:10],
}
with open(RESULTS_JSON, "w") as f:
    json.dump(best, f, indent=2)

print("\nTop configurations")
print(results.head(15).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", RESULTS_CSV)
print("Saved:", RESULTS_JSON)


=== proxy_reduce=none ===


Best so far: tuned_probe_prior_blend proxy=none prior=0.1 cmAP=0.9024 AUC=0.9011

=== proxy_reduce=max ===


Best so far: tuned_probe_prior_blend proxy=max prior=0.1 cmAP=0.9050 AUC=0.9026

=== proxy_reduce=mean ===


Best so far: tuned_probe_prior_blend proxy=max prior=0.1 cmAP=0.9050 AUC=0.9026

Top configurations
                 method  macro_auc  auc_classes  padded_cmap proxy_reduce  prior_blend  fitted_probes  proxy_targets
tuned_probe_prior_blend     0.9026           75       0.9050          max       0.1000            295              6
tuned_probe_prior_blend     0.9011           75       0.9048          max       0.2000            295              6
tuned_probe_prior_blend     0.9022           75       0.9045          max       0.0500            295              6
tuned_probe_prior_blend     0.8980           75       0.9044          max       0.3000            295              6
tuned_probe_prior_blend     0.9011           75       0.9037          max       0.0250            295              6
tuned_probe_prior_blend     0.9017           75       0.9031         mean       0.1000            295              6
tuned_probe_prior_blend     0.8998           75       0.9029         mean       0

## Result Notes

Record the best row in `experiments.md` after execution. The benchmark from `exp04` is `0.8934` macro AUC and `0.8993` padded cmAP.